# Machine Learning Bootcamp

Load Dependences:

In [65]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

## Step1: Identify problems that can be addressed with the dataset, Identify a question

### Question 1: Can we predict whether an insitution has a high retention rate (retain_value) based on its basic classification, and financial aid distribution.

An independent business metric of this question would be the financial aid distribution (the amount of aid given to students).

First we load the data an get a general sense of what we are seeing

In [66]:
cc = pd.read_csv("../data/cc_institution_details.csv")
cc.head()


,index,unitid,chronname,city,state,level,control,basic,hbcu,flagship,...,vsa_grad_after6_transfer,vsa_grad_elsewhere_after6_transfer,vsa_enroll_after6_transfer,vsa_enroll_elsewhere_after6_transfer,similar,state_sector_ct,carnegie_ct,counted_pct,nicknames,cohort_size
0,0,100654,Alabama A&M University,Normal,Alabama,4-year,Public,Masters Colleges and Universities--larger prog...,X,NaN,...,36.4,5.6,17.2,11.1,232937|100724|405997|113607|139533|144005|2285...,13,386,99.7|07,NaN,882.0
1,1,100663,University of Alabama at Birmingham,Birmingham,Alabama,4-year,Public,Research Universities--very high research acti...,NaN,NaN,...,NaN,NaN,NaN,NaN,196060|180461|201885|145600|209542|236939|1268...,13,106,56.0|07,UAB,1376.0
2,2,100690,Amridge University,Montgomery,Alabama,4-year,Private not-for-profit,Baccalaureate Colleges--Arts & Sciences,NaN,NaN,...,NaN,NaN,NaN,NaN,217925|441511|205124|247825|197647|221856|1353...,16,252,100.0|07,NaN,3.0
3,3,100706,University of Alabama at Huntsville,Huntsville,Alabama,4-year,Public,Research Universities--very high research acti...,NaN,NaN,...,0.0,0.0,0.0,0.0,232186|133881|196103|196413|207388|171128|1900...,13,106,43.1|07,UAH,759.0
4,4,100724,Alabama State University,Montgomery,Alabama,4-year,Public,Masters Colleges and Universities--larger prog...,X,NaN,...,NaN,NaN,NaN,NaN,100654|232937|242617|243197|144005|241739|2354...,13,386,88.0|07,ASU,1351.0


Now we must see the nature of the data that we are working with:

In [67]:
cc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3798 entries, 0 to 3797
Data columns (total 63 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   index                                 3798 non-null   int64  
 1   unitid                                3798 non-null   int64  
 2   chronname                             3798 non-null   object 
 3   city                                  3798 non-null   object 
 4   state                                 3798 non-null   object 
 5   level                                 3798 non-null   object 
 6   control                               3798 non-null   object 
 7   basic                                 3798 non-null   object 
 8   hbcu                                  94 non-null     object 
 9   flagship                              50 non-null     object 
 10  long_x                                3798 non-null   float64
 11  lat_y            

Now we should make the predictor and target variables categorical

In [68]:
cols =['basic']
cc[cols] = cc[cols].astype('category')
cc[['basic']].dtypes

basic    category
dtype: object

Now we should take another look at the the numbers in these categories to see what distributions we see and how we should group  everything

In [69]:
cc.basic.value_counts()

basic
Associates--Private For-profit                                                  517
Masters Colleges and Universities--larger programs                              386
Baccalaureate Colleges--Diverse Fields                                          343
Associates--Public Rural-serving Medium                                         289
Baccalaureate Colleges--Arts & Sciences                                         252
Masters Colleges and Universities--medium programs                              169
Associates--Public Rural-serving Large                                          128
Associates--Public Urban-serving Multicampus                                    125
Baccalaureate/Associates Colleges                                               124
Schools of art- music- and design                                               114
Associates--Public Rural-serving Small                                          111
Masters Colleges and Universities--smaller programs                   

Now we keep the bigger values and get rid of the smaller ones, keeeping about 5 major categories and putting everything else into other

In [70]:
top = ['Associates--Private For-profit', 'Masters Colleges and Universities--larger program', 'Baccalaureate Colleges--Diverse Fields', 'Associates--Public Rural-serving Medium', 'Baccalaureate Colleges--Arts & Sciences']

cc.basic = (cc.basic.apply(lambda x: x if x in top else "other")).astype('category')
cc.basic.value_counts()

basic
other                                      2397
Associates--Private For-profit              517
Baccalaureate Colleges--Diverse Fields      343
Associates--Public Rural-serving Medium     289
Baccalaureate Colleges--Arts & Sciences     252
Name: count, dtype: int64

We need to scale and normalize the aid_value

In [71]:
aid_value_sc = StandardScaler().fit_transform(cc[['aid_value']])
aid_value_n = MinMaxScaler().fit_transform(cc[['aid_value']])

Now we can implement one-hot encoding and normalize the categorical and numerical values

In [72]:
abc = list(cc.select_dtypes('number'))

cc[abc] = MinMaxScaler().fit_transform(cc[abc])

category_list = list(cc.select_dtypes('category'))

cc_1h = pd.get_dummies(cc, columns= category_list)

cc_1h

,index,unitid,chronname,city,state,level,control,hbcu,flagship,long_x,...,state_sector_ct,carnegie_ct,counted_pct,nicknames,cohort_size,basic_Associates--Private For-profit,basic_Associates--Public Rural-serving Medium,basic_Baccalaureate Colleges--Arts & Sciences,basic_Baccalaureate Colleges--Diverse Fields,basic_other
0,0.000000,0.000000,Alabama A&M University,Normal,Alabama,4-year,Public,X,NaN,0.790292,...,0.104348,0.746124,99.7|07,NaN,0.054289,False,False,False,False,True
1,0.000263,0.000024,University of Alabama at Birmingham,Birmingham,Alabama,4-year,Public,NaN,NaN,0.787680,...,0.104348,0.203488,56.0|07,UAB,0.084730,False,False,False,False,True
2,0.000527,0.000096,Amridge University,Montgomery,Alabama,4-year,Private not-for-profit,NaN,NaN,0.794572,...,0.130435,0.486434,100.0|07,NaN,0.000123,False,False,True,False,False
3,0.000790,0.000139,University of Alabama at Huntsville,Huntsville,Alabama,4-year,Public,NaN,NaN,0.789533,...,0.104348,0.203488,43.1|07,UAH,0.046709,False,False,False,False,True
4,0.001053,0.000187,Alabama State University,Montgomery,Alabama,4-year,Public,X,NaN,0.793252,...,0.104348,0.746124,88.0|07,ASU,0.083190,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3793,0.998947,0.963263,Grace College of Divinity,Fayetteville,North Carolina,4-year,Private not-for-profit,NaN,NaN,0.872837,...,0.365217,0.155039,NaN,NaN,0.000678,False,False,False,False,True
3794,0.999210,0.965468,John Paul the Great Catholic University,Escondido,California,4-year,Private not-for-profit,NaN,NaN,0.459168,...,0.634783,0.155039,NaN,NaN,0.001972,False,False,False,False,True
3795,0.999473,0.977658,Chamberlain College of Nursing-Missouri,St. Louis,Missouri,4-year,Private for-profit,NaN,NaN,0.748321,...,0.165217,0.155039,NaN,NaN,0.000431,False,False,False,False,True
3796,0.999737,0.998906,Minneapolis Media Institute,Edina,Minnesota,2-year,Private for-profit,NaN,NaN,0.716903,...,0.034783,0.155039,NaN,NaN,0.008874,False,False,False,False,True


Now we create a baseline:

In [73]:
cc_1h.boxplot(column= 'retain_value', vert = False, grid=False)
cc_1h.retain_value.describe()

count    3535.000000
mean        0.662319
std         0.170339
min         0.000000
25%         0.561000
50%         0.669000
75%         0.781000
max         1.000000
Name: retain_value, dtype: float64

Now we need to create a predictor that replaces the numeric version, and we use the bins value of corelatted to what we found when we plotted and described the "retain value" variable.

In [74]:
cc_1h['retain_value_f'] = pd.cut(cc_1h.retain_value, bins = [-1,0.78,1], labels = [0,1])

cc_1h

,index,unitid,chronname,city,state,level,control,hbcu,flagship,long_x,...,carnegie_ct,counted_pct,nicknames,cohort_size,basic_Associates--Private For-profit,basic_Associates--Public Rural-serving Medium,basic_Baccalaureate Colleges--Arts & Sciences,basic_Baccalaureate Colleges--Diverse Fields,basic_other,retain_value_f
0,0.000000,0.000000,Alabama A&M University,Normal,Alabama,4-year,Public,X,NaN,0.790292,...,0.746124,99.7|07,NaN,0.054289,False,False,False,False,True,0
1,0.000263,0.000024,University of Alabama at Birmingham,Birmingham,Alabama,4-year,Public,NaN,NaN,0.787680,...,0.203488,56.0|07,UAB,0.084730,False,False,False,False,True,1
2,0.000527,0.000096,Amridge University,Montgomery,Alabama,4-year,Private not-for-profit,NaN,NaN,0.794572,...,0.486434,100.0|07,NaN,0.000123,False,False,True,False,False,0
3,0.000790,0.000139,University of Alabama at Huntsville,Huntsville,Alabama,4-year,Public,NaN,NaN,0.789533,...,0.203488,43.1|07,UAH,0.046709,False,False,False,False,True,1
4,0.001053,0.000187,Alabama State University,Montgomery,Alabama,4-year,Public,X,NaN,0.793252,...,0.746124,88.0|07,ASU,0.083190,False,False,False,False,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3793,0.998947,0.963263,Grace College of Divinity,Fayetteville,North Carolina,4-year,Private not-for-profit,NaN,NaN,0.872837,...,0.155039,NaN,NaN,0.000678,False,False,False,False,True,0
3794,0.999210,0.965468,John Paul the Great Catholic University,Escondido,California,4-year,Private not-for-profit,NaN,NaN,0.459168,...,0.155039,NaN,NaN,0.001972,False,False,False,False,True,0
3795,0.999473,0.977658,Chamberlain College of Nursing-Missouri,St. Louis,Missouri,4-year,Private for-profit,NaN,NaN,0.748321,...,0.155039,NaN,NaN,0.000431,False,False,False,False,True,0
3796,0.999737,0.998906,Minneapolis Media Institute,Edina,Minnesota,2-year,Private for-profit,NaN,NaN,0.716903,...,0.155039,NaN,NaN,0.008874,False,False,False,False,True,1


In [75]:
prevalence = cc_1h.retain_value_f.value_counts()[1]/len(cc_1h.retain_value_f)

prevalence

0.23433385992627698

Now lets check if that number is accurate

In [76]:
print(cc_1h.retain_value_f.value_counts())
print(890/(890+2645))

retain_value_f
0    2645
1     890
Name: count, dtype: int64
0.25176803394625175


They look pretty close, now its time to clean up the data

In [77]:
cc_dt = cc_1h[['aid_value', 'basic_Associates--Private For-profit', 'basic_Associates--Public Rural-serving Medium', 'basic_Baccalaureate Colleges--Arts & Sciences', 'basic_Baccalaureate Colleges--Diverse Fields', 'basic_other', 'retain_value_f' ]].dropna()
cc_dt

,aid_value,basic_Associates--Private For-profit,basic_Associates--Public Rural-serving Medium,basic_Baccalaureate Colleges--Arts & Sciences,basic_Baccalaureate Colleges--Diverse Fields,basic_other,retain_value_f
0,0.165867,False,False,False,False,True,0
1,0.140338,False,False,False,False,True,1
2,0.054401,False,False,True,False,False,0
3,0.153878,False,False,False,False,True,1
4,0.168629,False,False,False,False,True,0
...,...,...,...,...,...,...,...
3792,0.106961,False,False,False,False,True,0
3793,0.110594,False,False,False,False,True,0
3794,0.220438,False,False,False,False,True,0
3795,0.072591,False,False,False,False,True,0


Now we can train and test our model

In [78]:
Train, Test = train_test_split(cc_dt, train_size= 2645, stratify = cc_dt.retain_value_f)

In [79]:
print(Train.shape)
print(Test.shape)

(2645, 7)
(889, 7)


In [80]:
Tune, Test = train_test_split(Test, train_size= .5, stratify= Test.retain_value_f)

Check prevalance in the groups

In [81]:
print(Train.retain_value_f.value_counts())
print(666/(1979+666))

retain_value_f
0    1979
1     666
Name: count, dtype: int64
0.25179584120982984


In [82]:
print(Tune.retain_value_f.value_counts())
print(112/(332+112))


retain_value_f
0    332
1    112
Name: count, dtype: int64
0.25225225225225223


## What do my instincts tell me about the data

My instincts tell me that the variables that I cleaned and used to train my model, were pretty clear indicators of the target variable, and I think that the model I created is going to be very accurate. I think that it will be able to address the problem, but I do think that there are many other variables that I could have chosen that may or may not be greater influences on my target variable. Furthermore the issue of there being a large number of na values also may hinder my model. This is my main concern.

## Now we can put this into a singular function

In [83]:
def preprocess_cc_data(url):
    
    cc = pd.read_csv(url)
    
    cols =['basic']
    cc[cols] = cc[cols].astype('category')
    
    top = ['Associates--Private For-profit', 'Masters Colleges and Universities--larger program', 'Baccalaureate Colleges--Diverse Fields', 'Associates--Public Rural-serving Medium', 'Baccalaureate Colleges--Arts & Sciences']
    cc.basic = (cc.basic.apply(lambda x: x if x in top else "other")).astype('category')
    
    aid_value_sc = StandardScaler().fit_transform(cc[['aid_value']])
    aid_value_n = MinMaxScaler().fit_transform(cc[['aid_value']])
    
    abc = list(cc.select_dtypes('number'))
    cc[abc] = MinMaxScaler().fit_transform(cc[abc])
    category_list = list(cc.select_dtypes('category'))
    cc_1h = pd.get_dummies(cc, columns= category_list)
    
    cc_1h['retain_value_f'] = pd.cut(cc_1h.retain_value, bins = [-1,0.78,1], labels = [0,1])
    
    cc_1h['retain_value_f'] = pd.cut(cc_1h.retain_value, bins = [-1,0.78,1], labels = [0,1])
    
    prevalence = cc_1h.retain_value_f.value_counts()[1]/len(cc_1h.retain_value_f)
    
    cc_dt = cc_1h[['aid_value', 'basic_Associates--Private For-profit', 'basic_Associates--Public Rural-serving Medium', 'basic_Baccalaureate Colleges--Arts & Sciences', 'basic_Baccalaureate Colleges--Diverse Fields', 'basic_other', 'retain_value_f' ]].dropna()
    
    Train, Test = train_test_split(cc_dt, train_size= 2645, stratify = cc_dt.retain_value_f)
    Tune, Test = train_test_split(Test, train_size= .5, stratify= Test.retain_value_f)
    
    return Train, Tune, Test, prevalence

url = "../data/cc_institution_details.csv"
Train, Test, Tune, prevalence = preprocess_cc_data(url)

## Question 2: Can we predict salary range by looking at factors such as gender, degree type, specialization, and mba percentile?

The Independent buissness metric of this question would be salary.

In [84]:
df = pd.read_csv("../data/job_placement.csv")
df.head()

,sl_no,gender,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status,salary
0,1,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed,270000.0
1,2,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed,200000.0
2,3,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed,250000.0
3,4,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed,NaN
4,5,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed,425000.0


Since we are trying to predict salary, we should remove all entires of people who have not been placed yet

In [85]:
df = df[df['status'] == 'Placed'].copy()

Now we need to check which types of variables we are working with

In [86]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 148 entries, 0 to 213
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           148 non-null    int64  
 1   gender          148 non-null    object 
 2   ssc_p           148 non-null    float64
 3   ssc_b           148 non-null    object 
 4   hsc_p           148 non-null    float64
 5   hsc_b           148 non-null    object 
 6   hsc_s           148 non-null    object 
 7   degree_p        148 non-null    float64
 8   degree_t        148 non-null    object 
 9   workex          148 non-null    object 
 10  etest_p         148 non-null    float64
 11  specialisation  148 non-null    object 
 12  mba_p           148 non-null    float64
 13  status          148 non-null    object 
 14  salary          148 non-null    float64
dtypes: float64(6), int64(1), object(8)
memory usage: 18.5+ KB


there are some vairbles we need to change in order to fit the operations we are about to do. First I will change the variables that should be casted as category variables

In [97]:
cols = ['gender','hsc_s','degree_t','workex','specialisation']
df[cols] = df[cols].astype('category')

Now lets take a look at the data types to make sure that everything is how we want it, before we start messing with variables

In [98]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 148 entries, 0 to 213
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   sl_no           148 non-null    float64 
 1   gender          148 non-null    category
 2   ssc_p           148 non-null    float64 
 3   ssc_b           148 non-null    object  
 4   hsc_p           148 non-null    float64 
 5   hsc_b           148 non-null    object  
 6   hsc_s           148 non-null    category
 7   degree_p        148 non-null    float64 
 8   degree_t        148 non-null    category
 9   workex          148 non-null    category
 10  etest_p         148 non-null    float64 
 11  specialisation  148 non-null    category
 12  mba_p           148 non-null    float64 
 13  status          148 non-null    object  
 14  salary          148 non-null    float64 
dtypes: category(5), float64(7), object(3)
memory usage: 14.1+ KB


In [99]:
df['gender'].value_counts()

gender
M    100
F     48
Name: count, dtype: int64

In [100]:
df['workex'].value_counts() 

workex
No     84
Yes    64
Name: count, dtype: int64

In [101]:
df['degree_t'].value_counts()

degree_t
Comm&Mgmt    102
Sci&Tech      41
Others         5
Name: count, dtype: int64

In [102]:
df['hsc_s'].value_counts()

hsc_s
Commerce    79
Science     63
Arts         6
Name: count, dtype: int64

In [103]:
df['specialisation'].value_counts()

specialisation
Mkt&Fin    95
Mkt&HR     53
Name: count, dtype: int64

In [104]:
df.head()

,sl_no,gender,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status,salary
0,0.000000,M,0.445545,Others,0.857051,Others,Commerce,0.057143,Sci&Tech,No,0.104167,Mkt&HR,0.251666,Placed,0.094595
1,0.004695,M,0.750743,Central,0.586729,Others,Science,0.613714,Sci&Tech,Yes,0.760417,Mkt&Fin,0.544884,Placed,0.000000
2,0.009390,M,0.396040,Central,0.366332,Central,Arts,0.228571,Comm&Mgmt,No,0.520833,Mkt&Fin,0.212466,Placed,0.067568
4,0.018779,M,0.910891,Central,0.485812,Central,Commerce,0.494286,Comm&Mgmt,No,0.975000,Mkt&Fin,0.122305,Placed,0.304054
7,0.032864,M,0.816832,Central,0.280990,Central,Science,0.285714,Sci&Tech,Yes,0.354167,Mkt&Fin,0.382595,Placed,0.070270


It looks like these categories are already grouped up properly, so now we can proceed with standarization and normalize for the integer and float values.

In [105]:
mba_p_sc = StandardScaler().fit_transform(df[['mba_p']])
mba_p_n = MinMaxScaler().fit_transform(df[['mba_p']])
abc = list(df.select_dtypes('number'))
df[abc] = MinMaxScaler().fit_transform(df[abc])
df

,sl_no,gender,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status,salary
0,0.000000,M,0.445545,Others,0.857051,Others,Commerce,0.057143,Sci&Tech,No,0.104167,Mkt&HR,0.251666,Placed,0.094595
1,0.004695,M,0.750743,Central,0.586729,Others,Science,0.613714,Sci&Tech,Yes,0.760417,Mkt&Fin,0.544884,Placed,0.000000
2,0.009390,M,0.396040,Central,0.366332,Central,Arts,0.228571,Comm&Mgmt,No,0.520833,Mkt&Fin,0.212466,Placed,0.067568
4,0.018779,M,0.910891,Central,0.485812,Central,Commerce,0.494286,Comm&Mgmt,No,0.975000,Mkt&Fin,0.122305,Placed,0.304054
7,0.032864,M,0.816832,Central,0.280990,Central,Science,0.285714,Sci&Tech,Yes,0.354167,Mkt&Fin,0.382595,Placed,0.070270
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209,0.981221,M,0.321782,Central,0.451675,Central,Commerce,0.257143,Comm&Mgmt,No,0.354167,Mkt&Fin,0.161113,Placed,0.021622
210,0.985915,M,0.782178,Others,0.665031,Others,Commerce,0.617143,Comm&Mgmt,No,0.854167,Mkt&Fin,0.866719,Placed,0.270270
211,0.990610,M,0.222772,Others,0.195648,Others,Science,0.457143,Sci&Tech,No,0.500000,Mkt&Fin,0.048608,Placed,0.101351
212,0.995305,M,0.445545,Others,0.344997,Others,Commerce,0.485714,Comm&Mgmt,Yes,0.187500,Mkt&Fin,0.679733,Placed,0.128378


Now its time to one-hot encode

In [106]:
category_list = list(df.select_dtypes('category'))
df_1h = pd.get_dummies(df, columns = category_list)
df_1h

,sl_no,ssc_p,ssc_b,hsc_p,hsc_b,degree_p,etest_p,mba_p,status,salary,...,hsc_s_Arts,hsc_s_Commerce,hsc_s_Science,degree_t_Comm&Mgmt,degree_t_Others,degree_t_Sci&Tech,workex_No,workex_Yes,specialisation_Mkt&Fin,specialisation_Mkt&HR
0,0.000000,0.445545,Others,0.857051,Others,0.057143,0.104167,0.251666,Placed,0.094595,...,False,True,False,False,False,True,True,False,False,True
1,0.004695,0.750743,Central,0.586729,Others,0.613714,0.760417,0.544884,Placed,0.000000,...,False,False,True,False,False,True,False,True,True,False
2,0.009390,0.396040,Central,0.366332,Central,0.228571,0.520833,0.212466,Placed,0.067568,...,True,False,False,True,False,False,True,False,True,False
4,0.018779,0.910891,Central,0.485812,Central,0.494286,0.975000,0.122305,Placed,0.304054,...,False,True,False,True,False,False,True,False,True,False
7,0.032864,0.816832,Central,0.280990,Central,0.285714,0.354167,0.382595,Placed,0.070270,...,False,False,True,False,False,True,False,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209,0.981221,0.321782,Central,0.451675,Central,0.257143,0.354167,0.161113,Placed,0.021622,...,False,True,False,True,False,False,True,False,True,False
210,0.985915,0.782178,Others,0.665031,Others,0.617143,0.854167,0.866719,Placed,0.270270,...,False,True,False,True,False,False,True,False,True,False
211,0.990610,0.222772,Others,0.195648,Others,0.457143,0.500000,0.048608,Placed,0.101351,...,False,False,True,False,False,True,True,False,True,False
212,0.995305,0.445545,Others,0.344997,Others,0.485714,0.187500,0.679733,Placed,0.128378,...,False,True,False,True,False,False,False,True,True,False
